# Dashboard: speed-dating match model

Run with `uv run python main.py app`. Everything here is loaded from
`artifacts/` (written by train.py). Nothing is trained: the three models are
loaded from disk and only used for inference on the held-out test events.


Run this notebook from the repository folder, after running `uv run python main.py train`.
Use **Run All** to load the saved models and launch the dashboard. No cell trains a model.
For the official entrypoint, use `uv run python main.py app`; `main.py` converts this notebook to a script.


In [ ]:
from __future__ import annotations

import json
import os

import gradio as gr
import joblib
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import torch
from plotly.subplots import make_subplots

from speeddating.data import ROOT
from speeddating.evaluation import (
    COST_FN, COST_FP, business_cost, calibration_table, confusion, cost_curve,
    cost_threshold, evaluate, seating_plan,
)
from speeddating.features import describe, group_of
from speeddating.torch_models import LogisticRegressionNet, predict_manual, predict_module

ART = ROOT / "artifacts"
if not (ART / "summary.json").exists():
    raise SystemExit("No artifacts found. Run `uv run python main.py train` first.")


## Load the trained models and data (no training here)


In [ ]:
with open(ART / "summary.json") as f:
    S = json.load(f)
FEATURES: list[str] = S["features"]
pre = joblib.load(ART / "preprocessor.joblib")
sk = joblib.load(ART / "sklearn_logreg.joblib")
manual = torch.load(ART / "torch_manual.pt", weights_only=True)
ckpt = torch.load(ART / "torch_module.pt", weights_only=True)
module = LogisticRegressionNet(ckpt["n_features"])
module.load_state_dict(ckpt["state_dict"])
module.eval()

test = pd.read_csv(ART / "test_pairs.csv")
dev = pd.read_csv(ART / "dev_pairs.csv")
Xs_test = pre.transform(test[FEATURES])
y_test = test["match"].to_numpy()

MODELS = {"sklearn": "scikit-learn", "manual": "PyTorch manual loop", "module": "PyTorch nn.Module + Adam"}
COLORS = {"sklearn": "#2a6fdb", "manual": "#e8833a", "module": "#2ca58d"}
PRED = {
    "sklearn": sk.predict_proba(Xs_test)[:, 1],
    "manual": predict_manual(manual, Xs_test),
    "module": predict_module(module, Xs_test),
}
COEF = pd.DataFrame({
    "feature": FEATURES,
    "sklearn": sk.coef_.ravel(),
    "manual": manual["w"].numpy().astype(float),
    "module": module.linear.weight.detach().numpy().ravel().astype(float),
})
INTERCEPT = {"sklearn": float(sk.intercept_[0]), "manual": float(manual["b"].item()),
             "module": float(module.linear.bias.item())}

# The loaded models must reproduce what train.py saw, or something is off.
saved = pd.read_csv(ART / "test_predictions.csv")
for k in PRED:
    gap = float(np.abs(PRED[k] - saved[f"p_{k}"].to_numpy()).max())
    if gap > 1e-5:
        raise SystemExit(f"Loaded {k} model does not reproduce training-time predictions (max gap {gap:.2e})")

T_STAR = cost_threshold(COST_FP, COST_FN)
TEST_WAVES = sorted(int(w) for w in test["wave"].unique())
N_DEV_EVENTS = len(S["split"]["train"]["waves"]) + len(S["split"]["validation"]["waves"])
LAYOUT = dict(template="plotly_white", margin=dict(l=60, r=20, t=50, b=50), font=dict(size=12))


def fmt_eur(x: float) -> str:
    return f"EUR {x:,.0f}"


## Tab 1: the three implementations


In [ ]:
def metrics_table() -> pd.DataFrame:
    rows = []
    for k, label in MODELS.items():
        r = evaluate(y_test, PRED[k], T_STAR)
        rows.append({"model": label, "ROC AUC": r["roc_auc"], "PR AUC": r["pr_auc"],
                     "log loss": r["log_loss"], "Brier": r["brier"],
                     f"cost at t={T_STAR:.2f}": fmt_eur(r["cost_eur"])})
    base = float(S["threshold"]["dev_base_rate"])
    for label, p in [("baseline: everyone gets the base rate", np.full(len(y_test), base)),
                     ("baseline: schedule everyone", np.ones(len(y_test)))]:
        r = evaluate(y_test, p, T_STAR)
        rows.append({"model": label, "ROC AUC": r["roc_auc"], "PR AUC": r["pr_auc"],
                     "log loss": r["log_loss"], "Brier": r["brier"],
                     f"cost at t={T_STAR:.2f}": fmt_eur(r["cost_eur"])})
    return pd.DataFrame(rows).round(4)


def agreement_table() -> pd.DataFrame:
    rows = []
    keys = list(MODELS)
    for i, a in enumerate(keys):
        for b in keys[i + 1:]:
            rows.append({
                "pair": f"{MODELS[a]} vs {MODELS[b]}",
                "max |coef diff|": f"{np.abs(COEF[a] - COEF[b]).max():.2e}",
                "|intercept diff|": f"{abs(INTERCEPT[a] - INTERCEPT[b]):.2e}",
                "max |prob diff| (test)": f"{np.abs(PRED[a] - PRED[b]).max():.2e}",
                "same decision at t*": f"{np.mean((PRED[a] >= T_STAR) == (PRED[b] >= T_STAR)):.1%}",
            })
    return pd.DataFrame(rows)


def fig_pred_vs_actual(n_bins: int = 8) -> go.Figure:
    fig = make_subplots(rows=1, cols=2, column_widths=[0.55, 0.45], subplot_titles=(
        "Predicted vs actual match rate (test events, equal-size bins)",
        "Predicted probability by what actually happened"))
    widths = {"sklearn": 6, "manual": 3.5, "module": 1.5}
    dashes = {"sklearn": "solid", "manual": "dash", "module": "dot"}
    for k, label in MODELS.items():
        cal = calibration_table(y_test, PRED[k], int(n_bins))
        fig.add_trace(go.Scatter(
            x=cal["mean_pred"], y=cal["match_rate"], mode="lines+markers", name=label,
            line=dict(color=COLORS[k], width=widths[k], dash=dashes[k]),
            error_y=dict(type="data", array=1.96 * cal["se"], visible=(k == "sklearn"), color=COLORS[k]),
            hovertemplate="mean predicted %{x:.3f}<br>observed %{y:.3f}<br>n=%{text}<extra>" + label + "</extra>",
            text=cal["n"]), row=1, col=1)
    fig.add_trace(go.Scatter(x=[0, 0.4], y=[0, 0.4], mode="lines", name="perfect calibration",
                             line=dict(color="#999", dash="dash", width=1)), row=1, col=1)
    for k, label in MODELS.items():
        for outcome, name in [(0, "no match"), (1, "match")]:
            fig.add_trace(go.Box(
                y=PRED[k][y_test == outcome], x=[f"{name}"] * int((y_test == outcome).sum()),
                name=label, legendgroup=k, showlegend=False, marker_color=COLORS[k],
                offsetgroup=k, boxpoints=False), row=1, col=2)
    fig.update_layout(boxmode="group", height=430, legend=dict(orientation="h", y=-0.2), **LAYOUT)
    fig.update_xaxes(title_text="mean predicted probability", range=[0, 0.4], row=1, col=1)
    fig.update_yaxes(title_text="observed match rate", range=[0, 0.4], row=1, col=1)
    fig.update_yaxes(title_text="predicted P(match)", row=1, col=2)
    return fig


def fig_coefficients() -> go.Figure:
    c = COEF.assign(desc=[describe(f) for f in COEF["feature"]],
                    group=[group_of(f) for f in COEF["feature"]]).sort_values("sklearn")
    fig = go.Figure()
    fig.add_trace(go.Bar(y=c["feature"], x=c["sklearn"], orientation="h", name="scikit-learn",
                         marker_color=COLORS["sklearn"], customdata=c[["desc", "group"]],
                         hovertemplate="%{y}: %{x:.4f}<br>%{customdata[0]}<br>group: %{customdata[1]}<extra></extra>"))
    for k, sym in [("manual", "diamond"), ("module", "x")]:
        fig.add_trace(go.Scatter(y=c["feature"], x=c[k], mode="markers", name=MODELS[k],
                                 marker=dict(color=COLORS[k], symbol=sym, size=8)))
    fig.update_layout(title="Coefficients on standardised features (log-odds per 1 SD)",
                      height=900, **LAYOUT)
    return fig


def fig_loss_curves() -> go.Figure:
    h = S["loss_history"]
    fig = go.Figure()
    for k in ("manual", "module"):
        gap = np.maximum(np.array(h[k]) - S["sklearn_objective_final"], 1e-9)
        fig.add_trace(go.Scatter(x=np.arange(len(gap)) * h["every"], y=gap, mode="lines",
                                 name=MODELS[k], line=dict(color=COLORS[k])))
    fig.update_layout(title="Training objective minus scikit-learn's minimum (log scale, floored at 1e-9)",
                      xaxis_title="epoch", yaxis_title="gap", yaxis_type="log", height=360, **LAYOUT)
    return fig


def lr_table() -> pd.DataFrame:
    t = pd.DataFrame(S["lr_search"])
    t["method"] = t["method"].map({"manual": "manual loop (plain gradient descent)", "module": "nn.Module + Adam"})
    return t.round(6)


## Tab 2: threshold and business cost


In [ ]:
def fig_confusion(cm: dict) -> go.Figure:
    z = [[cm["tn"], cm["fp"]], [cm["fn"], cm["tp"]]]
    text = [[f"{cm['tn']}<br>not seated, no match", f"{cm['fp']}<br>seated, no match (FP)"],
            [f"{cm['fn']}<br>missed match (FN)", f"{cm['tp']}<br>seated, matched (TP)"]]
    fig = go.Figure(go.Heatmap(z=z, x=["not seated", "seated"], y=["no match", "match"], text=text,
                               texttemplate="%{text}", colorscale="Blues", showscale=False))
    fig.update_layout(title=f"Confusion matrix on the {len(y_test)} test pairs", xaxis_title="decision",
                      yaxis_title="what happened", height=380, **LAYOUT)
    return fig


def threshold_view(model_key: str, t: float, c_fp: float, c_fn: float):
    c_fp = float(c_fp or 0)
    c_fn = float(c_fn or 0)
    p = PRED[model_key]
    cm = confusion(y_test, p, t)
    cost = business_cost(cm, c_fp, c_fn)
    everyone = business_cost(confusion(y_test, np.ones(len(p)), 0.5), c_fp, c_fn)
    nobody = business_cost(confusion(y_test, np.zeros(len(p)), 0.5), c_fp, c_fn)
    t_opt = cost_threshold(c_fp, c_fn) if (c_fp + c_fn) > 0 else 0.5
    curve = cost_curve(y_test, p, c_fp, c_fn)
    best = curve.loc[curve["cost_eur"].idxmin()]
    seated = cm["tp"] + cm["fp"]
    md = (
        f"### Cost at threshold {t:.3f}: **{fmt_eur(cost)}**\n"
        f"- schedule everyone (today's full rotation): {fmt_eur(everyone)}  \n"
        f"- schedule nobody / predict 'no match' for all: {fmt_eur(nobody)}  \n"
        f"- saving vs the better of the two: **{fmt_eur(min(everyone, nobody) - cost)}**"
        f"{' (negative: the model loses money at this threshold)' if cost > min(everyone, nobody) else ''}\n\n"
        f"Seated **{seated}** of {len(p)} pairs ({seated / len(p):.0%}); "
        f"they contain **{cm['tp']}** of the {int(y_test.sum())} real matches "
        f"(recall {cm['tp'] / max(1, y_test.sum()):.0%}, precision {cm['tp'] / max(1, seated):.0%}).\n\n"
        f"Decision rule for these costs: seat a pair when P(match) > c_fp / (c_fp + c_fn) = **{t_opt:.3f}**. "
        f"Lowest cost on the test curve: {fmt_eur(best['cost_eur'])} at t = {best['threshold']:.3f}."
    )
    fig = go.Figure()
    for k, label in MODELS.items():
        cc = cost_curve(y_test, PRED[k], c_fp, c_fn)
        fig.add_trace(go.Scatter(x=cc["threshold"], y=cc["cost_eur"], mode="lines", name=label,
                                 line=dict(color=COLORS[k], width=4 if k == model_key else 1.5)))
    fig.add_hline(y=everyone, line=dict(color="#999", dash="dot"), annotation_text="schedule everyone")
    fig.add_hline(y=nobody, line=dict(color="#555", dash="dash"), annotation_text="schedule nobody")
    fig.add_vline(x=t, line=dict(color="#d1495b"), annotation_text="your threshold")
    fig.add_vline(x=t_opt, line=dict(color="#333", dash="dot"), annotation_text="c_fp/(c_fp+c_fn)",
                  annotation_position="bottom right")
    fig.update_layout(title="Business cost on test events vs decision threshold", xaxis_title="threshold",
                      yaxis_title="cost (EUR)", xaxis_range=[0, max(0.5, t + 0.05, t_opt + 0.05)], height=420,
                      **LAYOUT)
    return fig_confusion(cm), md, fig


def threshold_for_costs(c_fp: float, c_fn: float) -> float:
    """When the costs change, move the slider to the cost-optimal threshold."""
    c_fp, c_fn = float(c_fp or 0), float(c_fn or 0)
    return round(cost_threshold(c_fp, c_fn), 3) if (c_fp + c_fn) > 0 else 0.5


## Tab 3: feature and target distributions


In [ ]:
ALL_COLS = FEATURES


def feature_view(col: str):
    d = dev[[col, "match"]].dropna()
    fig = make_subplots(specs=[[{"secondary_y": True}]])
    for outcome, name, color in [(0, "no match", "#9aa5b1"), (1, "match", "#d1495b")]:
        fig.add_trace(go.Histogram(x=d.loc[d["match"] == outcome, col], name=name, marker_color=color,
                                   opacity=0.65, histnorm="probability density", nbinsx=30), secondary_y=False)
    # match rate by quantile bin of the feature
    if d[col].nunique() > 6:
        bins = pd.qcut(d[col], 8, duplicates="drop")
        rate = d.groupby(bins, observed=True).agg(x=(col, "mean"), rate=("match", "mean"), n=("match", "size"))
    else:
        rate = d.groupby(col).agg(x=(col, "mean"), rate=("match", "mean"), n=("match", "size"))
    fig.add_trace(go.Scatter(x=rate["x"], y=rate["rate"], mode="lines+markers", name="match rate",
                             line=dict(color="#222"), text=rate["n"],
                             hovertemplate="%{x:.2f}: match rate %{y:.3f} (n=%{text})<extra></extra>"),
                  secondary_y=True)
    fig.update_layout(barmode="overlay", title=f"{col}: {describe(col)} (development events, before imputation)",
                      height=420, legend=dict(orientation="h", y=-0.2), **LAYOUT)
    fig.update_yaxes(title_text="density", secondary_y=False)
    fig.update_yaxes(title_text="match rate", secondary_y=True, rangemode="tozero", tickformat=".2f")
    coef = COEF.set_index("feature").loc[col, "sklearn"]
    md = (f"**{col}** - {describe(col)}. Group: *{group_of(col)}*. Missing before imputation: "
          f"{dev[col].isna().mean():.1%}. Model coefficient (per 1 SD): **{coef:+.4f}** "
          f"(odds x{np.exp(coef):.3f}).")
    return fig, md


def fig_target() -> go.Figure:
    allp = pd.concat([dev.assign(split="development"), test.assign(split="test")])
    g = allp.groupby("wave").agg(rate=("match", "mean"), her=("dec_W", "mean"), his=("dec_M", "mean"),
                                 n=("match", "size"), size=("event_size", "first")).reset_index()
    split = np.select([g["wave"].isin(S["split"]["test"]["waves"]), g["wave"].isin(S["split"]["validation"]["waves"])],
                      ["test", "validation"], "train")
    fig = go.Figure()
    for name, color in [("train", "#2a6fdb"), ("validation", "#e8833a"), ("test", "#2ca58d")]:
        m = split == name
        fig.add_trace(go.Bar(x=g["wave"][m], y=g["rate"][m], name=f"mutual match rate ({name})",
                             marker_color=color, customdata=np.c_[g["n"][m], g["size"][m]],
                             hovertemplate="wave %{x}: %{y:.3f}<br>%{customdata[0]} pairs, "
                                           "%{customdata[1]} partners each<extra></extra>"))
    fig.add_trace(go.Scatter(x=g["wave"], y=g["her"], mode="lines+markers", name="her yes rate",
                             line=dict(color="#d1495b")))
    fig.add_trace(go.Scatter(x=g["wave"], y=g["his"], mode="lines+markers", name="his yes rate",
                             line=dict(color="#555")))
    fig.update_layout(title=f"Target by event: mutual matches are rare ({S['data']['match_rate']:.1%}) and vary between nights",
                      xaxis_title="wave (event, in date order)", yaxis_title="rate", height=420,
                      legend=dict(orientation="h", y=-0.25), **LAYOUT)
    return fig


## Tab 4: event planner


In [ ]:
def planner_view(wave: int, share: float, model_key: str):
    ev = test[test["wave"] == int(wave)].reset_index(drop=True)
    p = PRED[model_key][(test["wave"] == int(wave)).to_numpy()]
    y = ev["match"].to_numpy()
    chosen = seating_plan(ev, p, share)
    rng = np.random.default_rng(0)
    rand = np.array([y[seating_plan(ev, rng.random(len(ev)), share)].sum() for _ in range(60)])
    perfect = y[seating_plan(ev, y + rng.random(len(ev)) * 1e-3, share)].sum()

    women = sorted(ev["iid_W"].unique())
    men = sorted(ev["iid_M"].unique())
    grid = np.full((len(women), len(men)), np.nan)
    wi = {w: i for i, w in enumerate(women)}
    mi = {m: j for j, m in enumerate(men)}
    for r, prob in zip(ev.itertuples(), p):
        grid[wi[r.iid_W], mi[r.iid_M]] = prob
    fig = go.Figure(go.Heatmap(z=grid, x=[f"M{m}" for m in men], y=[f"W{w}" for w in women],
                               colorscale="Blues", colorbar=dict(title="P(match)"),
                               hovertemplate="%{y} x %{x}: %{z:.3f}<extra></extra>"))
    seat = ev[chosen]
    fig.add_trace(go.Scatter(x=[f"M{m}" for m in seat["iid_M"]], y=[f"W{w}" for w in seat["iid_W"]],
                             mode="markers", name="seated by the model",
                             marker=dict(symbol="square-open", size=13, color="#e8833a", line=dict(width=2))))
    hit = ev[ev["match"] == 1]
    fig.add_trace(go.Scatter(x=[f"M{m}" for m in hit["iid_M"]], y=[f"W{w}" for w in hit["iid_W"]],
                             mode="markers", name="real mutual match",
                             marker=dict(symbol="circle", size=7, color="#d1495b")))
    fig.update_layout(title=f"Wave {wave}: predicted P(match) for every woman x man pair",
                      height=max(420, 22 * len(women) + 160), legend=dict(orientation="h", y=-0.12), **LAYOUT)
    k_w = int(np.ceil(share * len(men)))
    md = (f"### Wave {wave}: {len(women)} women x {len(men)} men, {int(y.sum())} real matches\n"
          f"Each woman meets {k_w} of {len(men)} men ({int(chosen.sum())} dates instead of {len(ev)}).\n\n"
          f"| plan | matches kept |\n|---|---|\n"
          f"| model ({MODELS[model_key]}) | **{int(y[chosen].sum())}** |\n"
          f"| random seating (mean of 60 plans) | {rand.mean():.1f} (range {rand.min()}-{rand.max()}) |\n"
          f"| perfect foresight | {int(perfect)} |\n\n"
          f"Share of random plans doing at least as well as the model: {np.mean(rand >= y[chosen].sum()):.0%}.")
    return fig, md


def planner_summary_md() -> str:
    rows = ["| share of the room each person meets | dates held | model plan | random plan (mean +/- sd) "
            "| perfect plan | random plans at least as good |", "|---|---|---|---|---|---|"]
    for share, r in S["planner"]["by_share"].items():
        rows.append(f"| {float(share):.0%} | {r['dates_held']} | **{r['model_plan']}** | "
                    f"{r['random_plan']:.1f} +/- {r['random_plan_sd']:.1f} | {r['perfect_plan']} | "
                    f"{r['share_random_draws_at_least_as_good']:.0%} |")
    n_matches = next(iter(S["planner"]["by_share"].values()))["all_matches"]
    return (f"All {len(TEST_WAVES)} test events together ({n_matches} real matches), scikit-learn model:\n\n"
            + "\n".join(rows))


## Tab 5: what physical attraction would add


In [ ]:
def fig_whatif() -> go.Figure:
    w = pd.DataFrame(S["attraction"]["whatif"])
    fig = make_subplots(rows=1, cols=2, subplot_titles=("Test ROC AUC", "Matches kept, half-room seating plan"))
    colors = ["#2a6fdb", "#c9c9c9", "#ababab", "#8d8d8d"]
    fig.add_trace(go.Bar(y=w["information"], x=w["roc_auc"], orientation="h", marker_color=colors,
                         text=w["roc_auc"].round(3), textposition="outside", showlegend=False), row=1, col=1)
    fig.add_trace(go.Bar(y=w["information"], x=w["planner_matches"], orientation="h", marker_color=colors,
                         text=w["planner_matches"], textposition="outside", showlegend=False), row=1, col=2)
    fig.update_yaxes(autorange="reversed", row=1, col=1)
    fig.update_yaxes(autorange="reversed", showticklabels=False, row=1, col=2)
    fig.update_xaxes(range=[0.45, 0.9], row=1, col=1)
    fig.update_layout(height=380, **LAYOUT)
    return fig


def fig_self_vs_received() -> go.Figure:
    pp = pd.DataFrame(S["attraction"]["people"])
    r = S["attraction"]["corr_self_rated_vs_received_looks"]
    rng = np.random.default_rng(0)
    fig = go.Figure()
    for g, name, color in [(0, "women", "#d1495b"), (1, "men", "#2a6fdb")]:
        d = pp[pp["gender"] == g]
        fig.add_trace(go.Scatter(x=d["self_attr"] + rng.normal(0, 0.12, len(d)), y=d["looks_received"],
                                 mode="markers", name=f"{name} (r = {r[name]:.2f})",
                                 marker=dict(color=color, size=6, opacity=0.55)))
    fig.update_layout(title="Self-rated attractiveness (sign-up) vs average rating received on the night",
                      xaxis_title="self-rated attractiveness (jittered)", yaxis_title="mean rating received",
                      height=420, **LAYOUT)
    return fig


def attraction_md() -> str:
    a = S["attraction"]
    pc = a["pair_level_corr"]
    side = a["one_sided_models"]
    return (
        "**The one looks signal we have before the event is a self-rating, and it barely tracks what "
        f"others see** (r = {a['corr_self_rated_vs_received_looks']['all']:.2f} overall, "
        f"{a['corr_self_rated_vs_received_looks']['women']:.2f} for women, "
        f"{a['corr_self_rated_vs_received_looks']['men']:.2f} for men). Meanwhile the attractiveness "
        f"rating a person receives correlates r = {a['corr_received_looks_vs_yes_received']:.2f} with the "
        "share of partners who say yes to them.\n\n"
        f"| pair-level correlation with a yes | her yes | his yes |\n|---|---|---|\n"
        f"| partner's self-rated looks (sign-up) | {pc['her_yes_vs_his_self_rated_looks']:.3f} "
        f"| {pc['his_yes_vs_her_self_rated_looks']:.3f} |\n"
        f"| how attractive they found the partner (scorecard) | {pc['her_yes_vs_her_rating_of_his_looks']:.3f} "
        f"| {pc['his_yes_vs_his_rating_of_her_looks']:.3f} |\n\n"
        f"Same features and model, predicting one side's yes instead of the mutual match: test AUC "
        f"**{side['her yes']['test_auc']:.3f}** for her decision, **{side['his yes']['test_auc']:.3f}** "
        "for his.\n\n"
        "Variants 2-4 use information from the night itself, so they are *not* deployable. They show the "
        "size of the gap. Note that the looks reputation raises AUC but keeps no extra matches in the "
        "seating plan: it tells you who is popular, and a fair plan gives everyone the same number of "
        "dates anyway. Only *pairwise* attraction (variant 3) moves the plan."
    )


## Tab 6: data size and split


In [ ]:
def fig_learning_curve() -> go.Figure:
    lc = pd.DataFrame(S["learning_curve"]["points"])
    sm = pd.DataFrame(S["learning_curve"]["summary"])
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=lc["pairs"], y=lc["val_auc"], mode="markers", name="one random draw of events",
                             marker=dict(color="#2a6fdb", size=5, opacity=0.3)))
    fig.add_trace(go.Scatter(x=sm["pairs"], y=sm["val_auc"], mode="lines+markers", name="validation AUC (mean)",
                             line=dict(color="#2a6fdb", width=3)))
    fig.add_trace(go.Scatter(x=sm["pairs"], y=sm["train_auc"], mode="lines+markers", name="training AUC (mean)",
                             line=dict(color="#999", width=2)))
    fig.add_hline(y=0.5, line=dict(color="#666", dash="dash"))
    fig.update_layout(title="Learning curve: training on 2 to 13 whole events, scored on the 3 validation events",
                      xaxis_title="training pairs (log scale)", xaxis_type="log", yaxis_title="ROC AUC",
                      height=420, **LAYOUT)
    return fig


def data_size_md() -> str:
    d, lc, lk, b = S["data"], S["learning_curve"], S["leakage_check"], S["bootstrap"]
    lo, hi = b["auc_ci"]["sklearn"]
    return (
        f"- **{d['date_records']:,}** date records = **{d['pairs']:,}** unique pairs, but only "
        f"**{d['people']}** people and **{d['events']}** events. Each person appears in "
        f"~{2 * d['pairs'] / d['people']:.0f} pairs, so rows are "
        "far from independent.\n"
        f"- **{d['matches']}** positives in total; {lc['positives_dev']} in the development events, i.e. "
        f"**{lc['events_per_variable_dev']:.1f} matches per feature** (rule of thumb for logistic regression: 10-20).\n"
        f"- Test set: {S['split']['test']['pairs']} pairs with {S['split']['test']['matches']} matches. "
        f"Bootstrap 95% interval for the test AUC: **{lo:.3f} - {hi:.3f}**. Differences between feature "
        "sets smaller than ~0.05 AUC cannot be told apart on this test set.\n"
        f"- Learning curve: about **{lc['auc_gain_per_doubling']:+.3f} AUC per doubling** of training pairs "
        "from 6 events up. The training-validation gap is still closing, so more events would help, slowly.\n"
        f"- Random-row split vs whole-event split (5-fold CV, same model): AUC "
        f"**{lk['random_rows_auc_mean']:.3f}** vs **{lk['by_event_auc_mean']:.3f}**. With random rows "
        f"{lk['share_eval_pairs_with_person_seen_in_training']:.0%} of evaluation pairs involve someone the "
        "model already saw in training."
    )


## Tab 7: pair explorer


In [ ]:
def women_in(wave: int) -> list:
    return sorted(int(w) for w in test.loc[test["wave"] == int(wave), "iid_W"].unique())


def men_for(wave: int, woman: int) -> list:
    m = (test["wave"] == int(wave)) & (test["iid_W"] == int(woman))
    return sorted(int(x) for x in test.loc[m, "iid_M"].unique())


def on_wave(wave):
    ws = women_in(wave)
    ms = men_for(wave, ws[0])
    return gr.Dropdown(choices=ws, value=ws[0]), gr.Dropdown(choices=ms, value=ms[0])


def on_woman(wave, woman):
    ms = men_for(wave, woman)
    return gr.Dropdown(choices=ms, value=ms[0])


def pair_view(wave, woman, man):
    m = ((test["wave"] == int(wave)) & (test["iid_W"] == int(woman)) & (test["iid_M"] == int(man))).to_numpy()
    if not m.any():
        return go.Figure(), "Pick a pair."
    i = int(np.flatnonzero(m)[0])
    row = test.iloc[i]
    contrib = Xs_test[i] * COEF["sklearn"].to_numpy()
    c = pd.DataFrame({"feature": FEATURES, "value": row[FEATURES].to_numpy(dtype=float),
                      "contribution": contrib}).assign(absval=lambda d: d["contribution"].abs())
    c = c.sort_values("absval", ascending=False).head(12).iloc[::-1]
    fig = go.Figure(go.Bar(y=c["feature"], x=c["contribution"], orientation="h",
                           marker_color=np.where(c["contribution"] > 0, "#2ca58d", "#d1495b"),
                           customdata=np.c_[c["value"], [describe(f) for f in c["feature"]]],
                           hovertemplate="%{y} = %{customdata[0]:.2f}<br>%{customdata[1]}"
                                         "<br>log-odds %{x:+.3f}<extra></extra>"))
    fig.update_layout(title="Top 12 contributions to the log-odds (scikit-learn): coefficient x standardised value",
                      xaxis_title="push towards match (+) or no match (-)", height=440, **LAYOUT)
    yes = lambda v: "yes" if v == 1 else "no"
    md = (f"### Woman {woman} x man {man} (wave {wave})\n"
          f"| | P(match) |\n|---|---|\n"
          + "".join(f"| {MODELS[k]} | **{PRED[k][i]:.4f}** |\n" for k in MODELS)
          + f"| base rate (development events) | {S['threshold']['dev_base_rate']:.4f} |\n\n"
          f"What happened: she said **{yes(row['dec_W'])}**, he said **{yes(row['dec_M'])}** -> "
          f"{'**mutual match**' if row['match'] == 1 else 'no match'}.\n\n"
          f"Intercept {INTERCEPT['sklearn']:+.3f}; sum of all contributions {contrib.sum():+.3f}. "
          "Missing answers were filled with the training median before scaling.")
    return fig, md


## Build the app


In [ ]:
INTRO = f"""
# Who will say yes to each other? A pre-event match model for speed-dating nights

An organiser gets everyone's sign-up survey before the night. If the night is too short for full rotation,
which pairs should meet? This dashboard serves a logistic regression trained **three ways** (scikit-learn,
a hand-written PyTorch loop, and the standard `nn.Module` + `torch.optim` workflow) on {N_DEV_EVENTS} past events and
evaluated on {len(TEST_WAVES)} later ones it never saw. All models are loaded from `artifacts/`; nothing is retrained here.

Business defaults: a seated pair that does not match costs **EUR {COST_FP:.0f}** (a wasted 4-minute slot for two
people), a match we fail to seat costs **EUR {COST_FN:.0f}**. That puts the decision threshold at
**{T_STAR:.2f}**. The costs are editable in tab 2.
"""


def build_demo() -> gr.Blocks:
    waves = TEST_WAVES
    w0 = women_in(waves[-1])[0]
    m0 = men_for(waves[-1], w0)[0]
    with gr.Blocks(title="Speed-dating match model") as demo:
        gr.Markdown(INTRO)
        with gr.Tab("1. Three implementations"):
            gr.Markdown(f"### Test events (waves {waves[0]}-{waves[-1]}): same split, same features, same objective")
            gr.Dataframe(value=metrics_table(), interactive=False)
            gr.Markdown("### Do they agree?")
            gr.Dataframe(value=agreement_table(), interactive=False)
            bins = gr.Slider(minimum=4, maximum=15, value=8, step=1, label="number of bins for the calibration plot")
            pva = gr.Plot(value=fig_pred_vs_actual(8))
            bins.change(fig_pred_vs_actual, inputs=bins, outputs=pva)
            gr.Plot(value=fig_coefficients())
            gr.Plot(value=fig_loss_curves())
            gr.Markdown(f"### Learning-rate search (fit on the training events, {S['epochs']} full-batch epochs)\n"
                        "Chosen = lowest final training objective without diverging. Adam uses a cosine "
                        "schedule that lowers the step size to 0 by the last epoch. "
                        f"Chosen: manual **{S['best_lr']['manual']}**, nn.Module **{S['best_lr']['module']}**; "
                        f"C = **{S['best_C']}** for all three.")
            gr.Dataframe(value=lr_table(), interactive=False)

        with gr.Tab("2. Threshold and cost"):
            with gr.Row():
                model_dd = gr.Dropdown(choices=[(v, k) for k, v in MODELS.items()], value="sklearn", label="model")
                thr = gr.Slider(minimum=0.0, maximum=1.0, value=round(T_STAR, 3), step=0.005,
                                label="decision threshold: seat the pair if P(match) >= threshold")
            with gr.Row():
                cfp = gr.Number(value=COST_FP, label="cost of a seated pair that does not match (EUR)", minimum=0)
                cfn = gr.Number(value=COST_FN, label="cost of a match we never seat (EUR)", minimum=0)
            cm0, md0, cc0 = threshold_view("sklearn", round(T_STAR, 3), COST_FP, COST_FN)
            with gr.Row():
                cm_plot = gr.Plot(value=cm0)
                cost_md = gr.Markdown(value=md0)
            curve_plot = gr.Plot(value=cc0)
            outs = [cm_plot, cost_md, curve_plot]
            for comp in (model_dd, thr):
                comp.change(threshold_view, inputs=[model_dd, thr, cfp, cfn], outputs=outs)
            for comp in (cfp, cfn):
                comp.change(threshold_for_costs, inputs=[cfp, cfn], outputs=thr).then(
                    threshold_view, inputs=[model_dd, thr, cfp, cfn], outputs=outs)

        with gr.Tab("3. Feature and target distributions"):
            gr.Plot(value=fig_target())
            feat = gr.Dropdown(choices=[(f"{c} - {describe(c)}", c) for c in ALL_COLS],
                               value="pref_fit_W", label="feature")
            f0, fmd0 = feature_view("pref_fit_W")
            feat_md = gr.Markdown(value=fmd0)
            feat_plot = gr.Plot(value=f0)
            feat.change(feature_view, inputs=feat, outputs=[feat_plot, feat_md])

        with gr.Tab("4. Event planner"):
            gr.Markdown("The decision the model is for: the night is too short for everyone to meet everyone. "
                        "The model's plan picks the dates with the highest total P(match), with the same "
                        "number of dates per person.\n\n" + planner_summary_md())
            with gr.Row():
                ev_dd = gr.Dropdown(choices=waves, value=waves[-1], label="test event (wave)")
                share = gr.Slider(minimum=0.2, maximum=1.0, value=0.5, step=0.05,
                                  label="share of the other side each person meets")
                pm_dd = gr.Dropdown(choices=[(v, k) for k, v in MODELS.items()], value="sklearn", label="model")
            pf0, pmd0 = planner_view(waves[-1], 0.5, "sklearn")
            plan_md = gr.Markdown(value=pmd0)
            plan_plot = gr.Plot(value=pf0)
            for comp in (ev_dd, share, pm_dd):
                comp.change(planner_view, inputs=[ev_dd, share, pm_dd], outputs=[plan_plot, plan_md])

        with gr.Tab("5. The missing piece: attraction"):
            gr.Markdown(attraction_md())
            gr.Plot(value=fig_whatif())
            gr.Plot(value=fig_self_vs_received())

        with gr.Tab("6. Data size and split"):
            gr.Markdown(data_size_md())
            gr.Plot(value=fig_learning_curve())
            gr.Dataframe(value=pd.DataFrame(S["c_search"]).round(4), interactive=False,
                         label="C search, leave-one-event-out on the development events")
            gr.Dataframe(value=pd.DataFrame(S["ablation"]).round(4), interactive=False,
                         label="Feature-group ablation (same scoring)")
            gr.Dataframe(value=pd.DataFrame(S["tried_and_dropped"]).round(4), interactive=False,
                         label="Feature blocks tried on top of the final set and dropped (same scoring)")

        with gr.Tab("7. Pair explorer"):
            with gr.Row():
                px_wave = gr.Dropdown(choices=waves, value=waves[-1], label="test event (wave)")
                px_w = gr.Dropdown(choices=women_in(waves[-1]), value=w0, label="woman (iid)")
                px_m = gr.Dropdown(choices=men_for(waves[-1], w0), value=m0, label="man (iid)")
            xf0, xmd0 = pair_view(waves[-1], w0, m0)
            px_md = gr.Markdown(value=xmd0)
            px_plot = gr.Plot(value=xf0)
            px_wave.change(on_wave, inputs=px_wave, outputs=[px_w, px_m]).then(
                pair_view, inputs=[px_wave, px_w, px_m], outputs=[px_plot, px_md])
            px_w.change(on_woman, inputs=[px_wave, px_w], outputs=px_m).then(
                pair_view, inputs=[px_wave, px_w, px_m], outputs=[px_plot, px_md])
            px_m.change(pair_view, inputs=[px_wave, px_w, px_m], outputs=[px_plot, px_md])
    return demo


demo = build_demo()

if __name__ == "__main__":
    # CI sets CI=true; don't try to open a browser there.
    demo.launch(inbrowser=not os.environ.get("CI"))
